# Chatbot or human agent: minutes to resolve a ticket

**Two-Sample Test for Means (Database Format)** · a worked example from the StatNex teaching guide

**The story.** A company logged how many minutes it took to resolve each of 27 support tickets: 18 handled by the chatbot and 9 by a human agent. The chatbot solves easy questions in a few minutes, but the tickets it cannot handle sit in a queue for a long time.

**The question.** Does the chatbot resolve tickets faster than a human agent?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/two_sample_mean_database.csv` (27 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Support,Minutes to resolve
1.0,Human agent,19.0
2.0,Chatbot,42.0
3.0,Chatbot,55.0
4.0,Chatbot,5.0
5.0,Chatbot,5.0
6.0,Human agent,13.0
7.0,Chatbot,3.0
8.0,Human agent,12.0
9.0,Human agent,15.0
10.0,Chatbot,8.0
11.0,Chatbot,6.0
12.0,Chatbot,4.0
13.0,Chatbot,30.0
14.0,Chatbot,9.0
15.0,Human agent,11.0
16.0,Chatbot,5.0
17.0,Chatbot,7.0
18.0,Chatbot,6.0
19.0,Chatbot,6.0
20.0,Human agent,17.0
21.0,Chatbot,7.0
22.0,Human agent,13.0
23.0,Chatbot,9.0
24.0,Chatbot,70.0
25.0,Chatbot,4.0
26.0,Human agent,16.0
27.0,Human agent,14.0

These cells save `cohen_d_interval.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile cohen_d_interval.py
"""Cohen's d with its noncentral-t interval, and Hedges' exact small-sample correction.

SciPy has the noncentral t distribution (stats.nct) but no function that inverts it for
an interval, so the inversion is written out (Steiger & Fouladi 1997). The interval is
for δ = t * scale: 1/sqrt(n) for one sample or paired differences, sqrt(1/n1 + 1/n2) for
two groups with the pooled standard deviation.
"""

import math

import numpy as np
from scipy import optimize, special, stats


def _limit(t, df, q):
    """The noncentrality under which the observed t sits at the q quantile of t(df, ncp).

    The cdf at t falls as the noncentrality rises, so the bracket opens outwards from t,
    doubling, until it holds the root.
    """
    def gap(ncp):
        return stats.nct.cdf(t, df, ncp) - q

    step = 1.0
    while gap(t - step) <= 0:
        step *= 2
    low = t - step
    step = 1.0
    while gap(t + step) >= 0:
        step *= 2
    return optimize.brentq(gap, low, t + step, xtol=1e-14, rtol=4 * np.finfo(float).eps,
                           maxiter=500)


def cohen_d_interval(d, df, scale, level):
    """(lower, upper): the level interval for d = t * scale."""
    t = d / scale
    return _limit(t, df, (1 + level) / 2) * scale, _limit(t, df, (1 - level) / 2) * scale


def hedges_j(df):
    """J = Γ(ν/2) / (sqrt(ν/2) Γ((ν - 1)/2)), Hedges (1981); g = J * d."""
    return math.exp(special.gammaln(df / 2) - special.gammaln((df - 1) / 2)) / math.sqrt(df / 2)

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# One row per case: a value column and a column saying which group the case is in.
# A row whose value is not a number, or whose group is blank, is dropped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
value, group = "Minutes to resolve", "Support"
confidence = 95 / 100
alpha = 5 / 100
frame = data[[group, value]].copy()
frame[value] = pd.to_numeric(frame[value], errors="coerce")
frame = frame.dropna()
# Group 1 and group 2 are the two group labels in sorted order.
levels = sorted(frame[group].unique())
values = [frame.loc[frame[group] == level, value].to_numpy(float) for level in levels]

## Descriptive statistics, with the mean rank

In [ ]:
# Every test below is repeated on the ranks of the pooled observations (ties share
# their mean rank): a robust alternative that keeps the t-test framework.
ranks = stats.rankdata(np.concatenate(values))
rank_values = [ranks[:len(values[0])], ranks[len(values[0]):]]
descriptive = pd.DataFrame({"n": [len(v) for v in values], "mean": [v.mean() for v in values],
                            "variance": [v.var(ddof=1) for v in values]}, index=levels)
descriptive["SD"] = np.sqrt(descriptive["variance"])
descriptive["SE"] = descriptive["SD"] / np.sqrt(descriptive["n"])
descriptive["mean rank"] = [v.mean() for v in rank_values]
print(descriptive)
ranked = pd.DataFrame({"n": [len(v) for v in rank_values],
                       "mean": [v.mean() for v in rank_values],
                       "variance": [v.var(ddof=1) for v in rank_values]}, index=levels)
samples = {"observed": descriptive, "ranked": ranked}

## Effect sizes, confidence intervals, t tests and power

In [ ]:
# Both t tests are written from (n, mean, variance), the same arithmetic for the
# values and for the ranks; stats.ttest_ind(x, y, equal_var=True / False) gives the
# same t. The pooled ("Student") test assumes equal variances; Welch's does not. Both
# compare means, which a few extreme values can drag around in skewed data: there the
# tests on the ranks describe the typical case better.
mean_difference = abs(descriptive["mean"].iloc[0] - descriptive["mean"].iloc[1])
effect, intervals, tests = {}, [], []
for label, d in samples.items():
    (n1, n2), (m1, m2), (v1, v2) = d["n"], d["mean"], d["variance"]
    difference = m1 - m2
    pooled_ss = (n1 - 1) * v1 + (n2 - 1) * v2
    # Cohen's d (SD on N) divides by the pooled SD over N, Cohen's d_s by the pooled SD
    # over N - 2; neither applies the small-sample correction, which Hedges' g_s adds.
    effect[label] = [abs(difference) / np.sqrt(pooled_ss / (n1 + n2)),
                     abs(difference) / np.sqrt(pooled_ss / (n1 + n2 - 2))]
    se_pooled = np.sqrt(pooled_ss / (n1 + n2 - 2)) * np.sqrt(1 / n1 + 1 / n2)
    se_welch = np.sqrt(v1 / n1 + v2 / n2)
    df_welch = (v1 / n1 + v2 / n2) ** 2 / ((v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1))
    for method, se, df in (("t test", se_pooled, n1 + n2 - 2), ("Welch", se_welch, df_welch)):
        t = difference / se
        p = 2 * stats.t.sf(abs(t), df)
        # Power: the chance a two-sided test at alpha rejects if the true effect equals
        # the one observed (non-central t with ncp = |t|). Computed from the observed effect,
        # it adds nothing to the P value; plan power from an effect that matters.
        critical = stats.t.ppf(1 - alpha / 2, df)
        power = stats.nct.sf(critical, df, abs(t)) + stats.nct.cdf(-critical, df, abs(t))
        tests.append([label, method, abs(t), df, p, stars(p), power])
        if label == "observed":
            # The interval is for |mean 1 - mean 2| and is always two-sided.
            half = stats.t.ppf(0.5 + confidence / 2, df) * se
            intervals.append([method, abs(difference) - half, abs(difference) + half])

effect = pd.DataFrame.from_dict(effect, orient="index", columns=["Cohen's d (SD on N)", "Cohen's d_s"])
intervals = pd.DataFrame(intervals, columns=["method", "lower", "upper"])
tests = pd.DataFrame(tests, columns=["data", "method", "t", "df", "P", "", "power"])
print(f"\nDifference of means |mean 1 - mean 2| = {mean_difference:.6g}")
print(effect)
print(f"\n{confidence:.0%} confidence interval for the difference")
print(intervals.to_string(index=False))
print()
print(tests.to_string(index=False))

## Two-sample Kolmogorov-Smirnov test

In [ ]:
# D is the largest gap between the two empirical distribution functions. The P value
# is from the limiting Kolmogorov distribution of sqrt(n1 n2 / (n1 + n2)) D;
# ks_2samp's own P (exact or method="asymp") uses finite-sample forms and differs.
ks = stats.ks_2samp(values[0], values[1])
ks_statistic = ks.statistic * np.sqrt(len(values[0]) * len(values[1]) / (len(values[0]) + len(values[1])))
ks_p = stats.kstwobign.sf(ks_statistic)
print(f"\nD = {ks.statistic:.6g}, KS = {ks_statistic:.6g}, P = {ks_p:.6g} {stars(ks_p)}")

## Effect sizes, with confidence intervals

In [ ]:
# Cohen's d_s = (mean 1 - mean 2) / pooled SD over N - 2: the Cohen's d_s above with its
# sign, and Student's t times sqrt(1/n1 + 1/n2). Its interval inverts the noncentral t
# at that t (Steiger & Fouladi 1997); Hedges' g_s = J * d_s removes d_s's small-sample
# bias, and its interval is d_s's times J. Both steps are in cohen_d_interval.py.
from cohen_d_interval import cohen_d_interval, hedges_j


def size(value):
    """Cohen's (1988) reading of |d|: 0.2 small, 0.5 medium, 0.8 large."""
    return ["negligible", "small", "medium", "large"][sum(abs(value) >= s for s in (0.2, 0.5, 0.8))]


(n1, n2), (m1, m2), (v1, v2) = descriptive["n"], descriptive["mean"], descriptive["variance"]
df = n1 + n2 - 2
d_s = (m1 - m2) / np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / df)
lower, upper = cohen_d_interval(d_s, df, np.sqrt(1 / n1 + 1 / n2), confidence)
j = hedges_j(df)
effect_sizes = pd.DataFrame(
    [[d_s, lower, upper, size(d_s)], [j * d_s, j * lower, j * upper, size(j * d_s)]],
    index=["Cohen's d_s", "Hedges' g_s"], columns=["value", "lower", "upper", "size"],
)
print()
print(effect_sizes)

## Normality: the Shapiro-Wilk test on each group

In [ ]:
normality = pd.DataFrame([[len(v), *stats.shapiro(v)] for v in values], index=levels,
                         columns=["n", "W", "P"])
print()
print(normality)

## What the result says

The mean time is 15.6 minutes for the chatbot and 14.4 for the human agent. Welch's t test, which does not assume equal spread in the two groups, gives P = 0.81. A P value is the chance of a difference at least this large if the true means were equal. So a difference in average time is not shown.

But the chatbot's times are very spread out: their standard deviation is 19.9 minutes against 2.6, and the Shapiro-Wilk test rejects a bell shape for them (P = 0.00002). The same test on ranks, which looks at the order of the tickets and not at the minutes, gives P = 0.002. The chatbot's mean rank is 11.5 against 19, so its typical ticket is solved sooner.

The answer depends on what "faster" means. The typical chatbot ticket is faster, but a few stuck tickets make its average no better.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.